# The math of overfitting: bias, variance and regularization

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Decomposing the error
The expected error of a model on new data splits into three parts:

$$E\big[(y-\hat{f}(x))^2\big]=\underbrace{\text{Bias}^2}_{\text{too simple}}+\underbrace{\text{Variance}}_{\text{too sensitive}}+\underbrace{\sigma^2}_{\text{noise}}$$

| Part | What it means | When it is large |
|---|---|---|
| Bias | Distance of the "average model" from the truth | The model is too simple (underfitting) |
| Variance | How much the model changes when the training data changes | The model is too complex (overfitting) |
| Noise $\sigma^2$ | The error built into the data | Always; it cannot be removed |

> [25 models, each trained on a different sample of 20 points. The orange line is their average.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_biasvar.png)

*25 models, each trained on a different sample of 20 points. The orange line is their average.*

| | Degree 1 | Degree 4 | Degree 12 |
|---|---|---|---|
| Bias | High | Low | Very low |
| Variance | Low | Medium | Very high |
| Training error | High | Low | Near zero |
| Test error | High | Lowest | High |

## L2 regularization (Ridge): shrinking the weights
$$L_{\text{ridge}}=\sum_i(y_i-\hat{y}_i)^2+\lambda\sum_j w_j^2$$

For one feature (no intercept) there is a closed form:

$$w_{\text{ridge}}=\frac{\sum x_iy_i}{\sum x_i^2+\lambda}$$

Example: if $\sum x_iy_i=20$ and $\sum x_i^2=10$:

| $\lambda$ | Calculation | $w$ |
|---|---|---|
| 0 | $20/10$ | 2.00 |
| 10 | $20/20$ | 1.00 |
| 90 | $20/100$ | 0.20 |
| 990 | $20/1000$ | 0.02 |

The weight shrinks but **never becomes exactly zero**. In gradient descent the L2 term multiplies the weight by $(1-2\eta\lambda)$ at every step, which is why it is also called **weight decay**.

## L1 regularization (Lasso): zeroing weights
$$L_{\text{lasso}}=\sum_i(y_i-\hat{y}_i)^2+\lambda\sum_j\lvert w_j\rvert$$

Its solution is "soft thresholding": pull the weight toward zero by t, and if it would cross zero, leave it there:

$$w_{\text{lasso}}=\mathrm{sign}(w_{\text{ols}})\cdot\max\big(0,\ \lvert w_{\text{ols}}\rvert-t\big),\qquad t=\frac{\lambda}{2\sum x_i^2}$$

| $w_{\text{ols}}$ | $t=0.5$ | $t=1.5$ | $t=2.5$ |
|---|---|---|---|
| 2.0 | 1.5 | 0.5 | 0 |
| −0.8 | −0.3 | 0 | 0 |
| 0.3 | 0 | 0 | 0 |

Small weights become **exactly zero**, so Lasso performs feature selection.

## Why does L1 give zeros and L2 does not?
> [The L1 region is a diamond with its corners on the axes; the L2 region is a circle.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/l1_l2.png)

*The L1 region is a diamond with its corners on the axes; the L2 region is a circle.*

| | L2 | L1 |
|---|---|---|
| Shape of the constraint | Circle | Diamond |
| Where the loss contours touch it | Usually somewhere on the arc | Usually at a corner (one weight = 0) |
| Derivative of the penalty near zero | $2\lambda w\to0$: the push fades | $\pm\lambda$: a constant push all the way to zero |
| Bayesian prior | Normal | Laplace |

## Other ways to regularize
| Method | The mathematical idea |
|---|---|
| Early stopping | Stop training when the validation error rises; acts much like L2 |
| Dropout | Each neuron is dropped with probability p; like averaging many sub-networks |
| More data | More samples reduce the variance (standard error $\propto1/\sqrt{n}$) |
| Bagging | Averaging M independent models divides the variance by M |



In [ ]:
import numpy as np

rng = np.random.default_rng(0)
truth = lambda x: np.sin(2 * np.pi * x)
x_test = np.linspace(0.1, 0.9, 50)

# Bias and variance measured by training 300 models on 300 different samples
print("degree   bias^2   variance   total (+ noise 0.09)")
for degree in [1, 4, 12]:
    preds = []
    for _ in range(300):
        x = rng.uniform(0, 1, 40); y = truth(x) + rng.normal(0, 0.3, 40)
        preds.append(np.polyval(np.polyfit(x, y, degree), x_test))
    preds = np.array(preds)
    bias2 = np.mean((preds.mean(axis=0) - truth(x_test)) ** 2)
    var = np.mean(preds.var(axis=0))
    print(f"{degree:5d}   {bias2:7.4f}   {var:8.4f}   {bias2 + var + 0.09:.4f}")

# Ridge with one feature: w = sum(xy) / (sum(x^2) + lambda)
for lam in [0, 10, 90, 990]:
    print(f"lambda = {lam:4d}:  w = {20 / (10 + lam):.2f}")

# Lasso: soft thresholding
soft = lambda w, t: np.sign(w) * np.maximum(0, np.abs(w) - t)
w_ols = np.array([2.0, -0.8, 0.3])
for t in [0.5, 1.5, 2.5]:
    print(f"t = {t}:", soft(w_ols, t))

# The same effect with scikit-learn on 10 features, only 3 of which matter
from sklearn.linear_model import Lasso, LinearRegression, Ridge
X = rng.normal(size=(60, 10)); y = 3 * X[:, 0] - 2 * X[:, 1] + X[:, 2] + rng.normal(0, 0.5, 60)
for name, model in [("OLS  ", LinearRegression()), ("Ridge", Ridge(alpha=20)), ("Lasso", Lasso(alpha=0.2))]:
    w = model.fit(X, y).coef_
    print(name, w.round(2), "  zeros:", int(np.sum(np.abs(w) < 1e-8)))

# Averaging M independent models divides the variance by M
single = rng.normal(0, 1, (10_000, 25))
print("variance of one model:", single[:, 0].var().round(3), "  of the average of 25:", single.mean(axis=1).var().round(3))

### Exercise

If $\sum x_iy_i=30$ and $\sum x_i^2=15$, compute the Ridge weight for $\lambda=0,\ 15,\ 45$.

In [ ]:
# Your answer here

### Exercise

Apply soft thresholding with $t=1$ to the weights $(3,\ -0.5,\ 1,\ -2)$.

In [ ]:
# Your answer here